In [0]:
from pyspark.sql.functions import col, count, month, year, to_date, concat, lit, when, last_day, lpad
from pyspark.sql import Window
import json
import os

# Paths (Updated to avoid DBFS issues)
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/Y02E10_vacancies_data_2015_16"
checkpoint_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/checkpoints.json"

# Convert DBFS path to local path for JSON handling
local_checkpoint_path = "/dbfs" + checkpoint_path.lstrip("dbfs:")

# Load all files at once (parallel processing)
df = spark.read.parquet("dbfs:/mnt/light/data/world_bank_global_postings/version=3/2025/02/26/00_26_56/*.parquet")

# Filter for US & jobs posted after 2017-01-01
df_us = df.filter((col('nation_name') == "United States") & (col('posted') >= "2015-01-01") & (col('posted') < '2017-01-01'))

# Define keywords for solar-related jobs
keywords = [
    "sea energy", "sea power", "Energy from the Sea", "power from the sea", "wave energy", "wave power", "salinity gradient energy", "salinity gradient power", "solar thermal energy", "solar thermal power", "solar tower", "heat exchange system", "thermal power", "thermal energy", "thermal engine", "photovoltaic energy", "photovoltaic power", "PV energy", "PV power", "photovoltaic cell", "PV cell", "photovoltaic system", "PV system", "thermal-PV hybrid", "thermal-photovoltaic hybrid", "thermal photovoltaic hybrid", "thermal PV hybrid", "solar cell", "wind energy", "wind power", "wind turbine", "solar energy", "solar power", "photovoltaic effect", "PV effect", "solar radiation", "heat radiation", "sea energies", "sea powers", "energies from the sea", "power from the sea", "wave energies", "wave powers", "salinity gradient energies", "salinity gradient powers", "solar thermal energies", "solar thermal powers", "solar towers", "heat exchange systems", "thermal powers", "thermal energies", "thermal engines", "photovoltaic energies", "photovoltaic powers", "PV energies", "PV powers", "photovoltaic cells", "PV cells", "photovoltaic systems", "PV systems", "thermal-PV hybrids", "thermal-photovoltaic hybrids", "thermal photovoltaic hybrids", "thermal PV hybrids", "solar cells", "wind energies", "wind powers", "wind turbines", "solar energies", "solar powers", "photovoltaic effects", "PV effects", "solar radiations", "heat radiations"
]
regex_values = "|".join(keywords)

# Filter for solar-related jobs
df = df_us.filter(col("body").rlike(regex_values))

new_df = (
    df.select("posted", "employment_type", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .withColumn("month", lpad(month(col("posted")).cast("string"), 2, "0"))
    .withColumn("year", year(col("posted")))
    .groupBy("year", "month", "laa_admin_area_2", "min_years_experience", "isco_08_1")
    .agg(
        count(when(col("employment_type") == 1, True)).alias("full_time_count"),
        count(when(col("employment_type") == 2, True)).alias("part_time_count"),
        count("*").alias("total_count")
    )
    .withColumnRenamed("laa_admin_area_2", "county_id")
    .withColumnRenamed("isco_08_1", "career_type")
    .withColumn("start_date", to_date(concat(col("year"), lit("-"), col("month"), lit("-01")), "yyyy-MM-dd"))
    .withColumn("end_date", last_day(col("start_date")))
)

# Append data as CSV (instead of Parquet)
new_df.write.format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("sep", ",") \
    .save(output_path)

In [0]:
# Load processed data
output_path = "/Volumes/prd_datascience_onlinejobpostings/volumes/onlinejobpostings/joep_keuzenkamp/Y02E10_vacancies_data_2015_16"

df = spark.read.csv(output_path)

display(df)

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9
year,month,county_id,min_years_experience,career_type,full_time_count,part_time_count,total_count,start_date,end_date
2015,05,US6081,8,3,1,0,1,2015-05-01,2015-05-31
2015,05,US6081,4,3,3,0,3,2015-05-01,2015-05-31
2015,06,US34003,null,2,1,0,4,2015-06-01,2015-06-30
2015,01,US12031,6,2,1,0,1,2015-01-01,2015-01-31
2016,09,US39113,null,2,1,0,1,2016-09-01,2016-09-30
2015,02,US38003,2,3,1,0,1,2015-02-01,2015-02-28
2015,02,US36087,0,2,1,0,1,2015-02-01,2015-02-28
2015,02,US6075,null,2,7,0,7,2015-02-01,2015-02-28
2016,07,US27053,10,null,2,0,2,2016-07-01,2016-07-31
